In [ ]:
import os
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "wandb_api_key"
wandb_key = UserSecretsClient().get_secret(secret_label)

In [ ]:
import wandb

wandb.login(key=wandb_key)

## Prepare data

### Build a metadata dataframe

In [ ]:
dataset_root = "/kaggle/input/datasets/suranjandas1990/forest-loss-dataset/dataset"

records = []

regions = ["Meghalaya_2021_2023", "Nagaland_2021_2023"]

for region in regions:

    label_dir = os.path.join(dataset_root, region, "label")

    for fname in tqdm(os.listdir(label_dir)):

        path = os.path.join(label_dir, fname)

        mask = np.load(path)

        change_pixels = mask.sum()
        total_pixels = mask.size

        change_ratio = change_pixels / total_pixels

        records.append({
            "region": region,
            "file": fname,
            "label_path": path,
            "change_pixels": change_pixels,
            "change_ratio": change_ratio
        })

df = pd.DataFrame(records)

In [ ]:
print("Total samples:", len(df))
df.head()

### Create stratification bins

To handle the high class imbalance that is inherent in the forest change detection data, the continuous change ratio of the patches was binned into meaningful categorical bins that indicate different classes of deforestation intensity, i.e., no change, very sparse, sparse, and moderate to large change. This helps in the structured understanding of the data distribution, as the majority of the data points indicate minimal or no change in the forest area. In addition, a composite stratification key was created by combining the geographic region with the change category, ensuring that the data splitting operations, i.e., the division of the data into training, validation, and test sets, account for diversity in the data as well as the class imbalance problem. This is crucial in avoiding bias in the data, which is necessary for the robustness of the deep learning model.

In [ ]:
bins = [0, 1e-6, 0.005, 0.02, 1]
labels = [
    "no_change",
    "very_sparse",
    "sparse",
    "moderate_large"
]

df["change_bin"] = pd.cut(
    df["change_ratio"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df["stratify_key"] = df["region"] + "_" + df["change_bin"].astype(str)

In [ ]:
df.sample(5)['stratify_key']

In [ ]:
df["change_bin"].value_counts() / len(df)

In [ ]:
import matplotlib.pyplot as plt

# Data from user
categories = ["very_sparse", "sparse", "moderate_large", "no_change"]
values = [0.356006, 0.347871, 0.184493, 0.111630]

plt.figure()
plt.bar(categories, values)
plt.xlabel("Change Category")
plt.ylabel("Proportion")
plt.title("Distribution of Change Categories")

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Stratified train / val / test split

The dataset was divided into subsets for training, validation, and testing. This was done using a stratified sampling strategy. The strategy ensured that each subset had an almost balanced representation of both geographic regions and levels of forest change intensity. This was achieved through using a composite stratification key, where both region and category of change were considered. The dataset was divided into 70%, 15%, and 15% subsets for training, validation, and testing, respectively. This strategy eliminates sampling biases and allows each subset to have an almost balanced representation of no change, sparse, and high change patches.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["stratify_key"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["stratify_key"],
    random_state=42
)

print(len(train_df), len(val_df), len(test_df))

### Save the splits

In [ ]:
train_df.to_csv("train_split.csv", index=False)
val_df.to_csv("val_split.csv", index=False)
test_df.to_csv("test_split.csv", index=False)

In [ ]:
# # code to calculate train data mean and std
# train_df = pd.read_csv("/kaggle/working/train_split.csv")
# root_dir = "/kaggle/input/datasets/suranjandas1990/forest-loss-dataset/dataset"

# sum_c = torch.zeros(6)
# sum_sq_c = torch.zeros(6)
# num_pixels = 0

# for _, row in tqdm(train_df.iterrows(), total=len(train_df)):

#     region = row["region"]
#     fname = row["file"]

#     t1 = np.load(f"{root_dir}/{region}/t1/{fname}", mmap_mode="r")
#     t2 = np.load(f"{root_dir}/{region}/t2/{fname}", mmap_mode="r")

#     t1 = torch.tensor(t1, dtype=torch.float32)
#     t2 = torch.tensor(t2, dtype=torch.float32)

#     img = torch.cat([t1, t2], dim=0)  # shape (12, H, W)

#     sum_c += img.sum(dim=[1,2])[:6] + img.sum(dim=[1,2])[6:]
#     sum_sq_c += (img**2).sum(dim=[1,2])[:6] + (img**2).sum(dim=[1,2])[6:]

#     num_pixels += 2 * img.shape[1] * img.shape[2]

# mean = sum_c / num_pixels
# std = torch.sqrt(sum_sq_c / num_pixels - mean**2)

# print("Mean:", mean.tolist())
# print("Std:", std.tolist())

### Dataset and Dataloader

A custom PyTorch dataset class has been developed that facilitates the memory-efficient loading and preprocessing of multi-temporal satellite images. In each batch, there are two types of input images: bi-temporal images (t1 and t2) derived from Sentinel-1 and Sentinel-2 images, and a corresponding binary mask of forest loss. Normalization of input images is done using precomputed mean and standard deviation. NumPy’s memory-mapped arrays are used for memory-efficient loading of images. This allows the model to be trained on large-scale images. The dataset is designed in such a way that it is suitable for Siamese networks, where the model is trained to directly predict change.

In [ ]:
data_mean = torch.tensor([0.0187, 0.0389, 0.0231, 0.2954, -8.4814, -14.8727]).view(6,1,1)
data_std = torch.tensor([0.0137, 0.0186, 0.0188, 0.0970, 2.2510, 2.2649]).view(6,1,1)

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np

class ChangeDataset(Dataset):

    def __init__(self, dataframe, root_dir, mean=data_mean, std=data_std):

        self.df = dataframe.reset_index(drop=True)
        self.root = root_dir

        # dataset mean/std (replace with values computed from your dataset)
        self.mean = mean
        self.std  = std

    def __len__(self):
        return len(self.df)

    def normalize(self, x):
        return (x - self.mean) / (self.std + 1e-6)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        region = row["region"]
        fname = row["file"]

        t1 = np.load(f"{self.root}/{region}/t1/{fname}", mmap_mode="r")
        t2 = np.load(f"{self.root}/{region}/t2/{fname}", mmap_mode="r")
        label = np.load(f"{self.root}/{region}/label/{fname}")

        t1 = torch.tensor(t1, dtype=torch.float32)
        t2 = torch.tensor(t2, dtype=torch.float32)

        t1 = self.normalize(t1)
        t2 = self.normalize(t2)

        label = torch.tensor(label, dtype=torch.float32).unsqueeze(0)

        return t1, t2, label

In [ ]:
train_df = pd.read_csv("/kaggle/working/train_split.csv")
val_df = pd.read_csv("/kaggle/working/val_split.csv")

root_dir = "/kaggle/input/datasets/suranjandas1990/forest-loss-dataset/dataset"

train_dataset = ChangeDataset(train_df, root_dir)
val_dataset = ChangeDataset(val_df, root_dir)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
                          num_workers=4, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False,
                        num_workers=4, pin_memory=True)

## Model Build and Training

A Siamese U-Net structure, with the addition of Atrous Spatial Pyramid Pooling (ASPP), has been adopted for forest change detection. The proposed network works on pairs of bi-temporal satellite images, using a shared encoder network for hierarchical feature extraction. This ensures that the features are represented in a consistent manner for all time steps. The change information is then derived by finding the absolute difference between the encoded features of the two input images. The ASPP layer has been used for capturing contextual information, which is useful for detecting small-scale as well as large-scale deforestation patterns. The decoder uses skip connections for reconstructing the spatial information, fusing features from both input images. The proposed network has been trained using a hybrid loss function, consisting of Binary Cross-Entropy and Dice Loss. The network has been evaluated using the F1-score and Intersection over Union (IoU) metrics, which are effective for measuring the accuracy of the detection results.

In [ ]:
t1, t2, label = train_dataset[0]
t1.shape, t2.shape, label.shape

In [ ]:
class DoubleConv(nn.Module):

    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

In [ ]:
class ASPP(nn.Module):

    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv1 = nn.Conv2d(in_ch, out_ch, 1)

        self.conv2 = nn.Conv2d(in_ch, out_ch, 3, padding=6, dilation=6)
        self.conv3 = nn.Conv2d(in_ch, out_ch, 3, padding=12, dilation=12)
        self.conv4 = nn.Conv2d(in_ch, out_ch, 3, padding=18, dilation=18)

        self.project = nn.Conv2d(out_ch * 4, out_ch, 1)

    def forward(self, x):

        f1 = self.conv1(x)
        f2 = self.conv2(x)
        f3 = self.conv3(x)
        f4 = self.conv4(x)

        x = torch.cat([f1, f2, f3, f4], dim=1)

        return self.project(x)

In [ ]:
class Encoder(nn.Module):

    def __init__(self, in_channels=6):
        super().__init__()

        self.conv1 = DoubleConv(in_channels, 64)
        self.conv2 = DoubleConv(64, 128)
        self.conv3 = DoubleConv(128, 256)
        self.conv4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

    def forward(self, x):

        f1 = self.conv1(x)
        p1 = self.pool(f1)

        f2 = self.conv2(p1)
        p2 = self.pool(f2)

        f3 = self.conv3(p2)
        p3 = self.pool(f3)

        f4 = self.conv4(p3)
        p4 = self.pool(f4)

        return [f1, f2, f3, f4], p4

In [ ]:
class UpBlock(nn.Module):

    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()

        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)

        self.conv = DoubleConv(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat([x, skip], dim=1)

        return self.conv(x)

In [ ]:
class SiameseUNet_ASPP(nn.Module):

    def __init__(self, in_channels=6):
        super().__init__()

        self.encoder = Encoder(in_channels)

        self.aspp = ASPP(512, 512)

        self.up4 = UpBlock(512, 1024, 512)
        self.up3 = UpBlock(512, 512, 256)
        self.up2 = UpBlock(256, 256, 128)
        self.up1 = UpBlock(128, 128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, t1, t2):

        feat1, bottleneck1 = self.encoder(t1)
        feat2, bottleneck2 = self.encoder(t2)

        x = torch.abs(bottleneck1 - bottleneck2)

        x = self.aspp(x)

        s4 = torch.cat([feat1[3], feat2[3]], dim=1)
        s3 = torch.cat([feat1[2], feat2[2]], dim=1)
        s2 = torch.cat([feat1[1], feat2[1]], dim=1)
        s1 = torch.cat([feat1[0], feat2[0]], dim=1)

        x = self.up4(x, s4)
        x = self.up3(x, s3)
        x = self.up2(x, s2)
        x = self.up1(x, s1)

        out = self.final(x)

        return out

In [ ]:
class DiceLoss(nn.Module):

    def forward(self, pred, target):

        pred = torch.sigmoid(pred)

        smooth = 1e-6

        intersection = (pred * target).sum()

        union = pred.sum() + target.sum()

        dice = (2 * intersection + smooth) / (union + smooth)

        return 1 - dice

In [ ]:
bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([20]).cuda())
dice = DiceLoss()

def loss_fn(pred, target):
    return 0.5*bce(pred,target) + 0.5*dice(pred,target)

In [ ]:
def compute_metrics(pred, target):

    pred = torch.sigmoid(pred)
    pred = (pred > 0.3).float()

    tp = (pred * target).sum()
    fp = (pred * (1-target)).sum()
    fn = ((1-pred) * target).sum()

    precision = tp/(tp+fp+1e-6)
    recall = tp/(tp+fn+1e-6)

    f1 = 2*precision*recall/(precision+recall+1e-6)

    iou = tp/(tp+fp+fn+1e-6)

    return f1.item(), iou.item()

### Training Loop

In [ ]:
device = "cuda"

model = SiameseUNet_ASPP()

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)

model = model.cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
model

In [ ]:
wandb.init(
    project="Forest loss detection",
    name="unet_aspp_dice_bce",
    config={
        "model": "UNet_ASPP",
        "loss": "Dice+BCE",
        "lr": 1e-4,
        "batch_size": 16,
        "epochs": 50
    }
)

In [ ]:
wandb.watch(model, log="all", log_freq=100)

In [ ]:
best_iou = 0
patience = 5
epochs_without_improvement = 0

scaler = GradScaler(device="cuda")

for epoch in range(50):

    model.train()
    train_loss = 0

    for t1, t2, label in tqdm(train_loader):

        t1 = t1.to(device)
        t2 = t2.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        with autocast(device_type="cuda"):
            pred = model(t1, t2)
            loss = loss_fn(pred, label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)
    print("Train loss:", train_loss)

    model.eval()

    f1_scores = []
    ious = []

    with torch.no_grad():

        for t1, t2, label in val_loader:

            t1 = t1.to(device)
            t2 = t2.to(device)
            label = label.to(device)

            with autocast(device_type="cuda"):
                pred = model(t1, t2)

            f1, iou = compute_metrics(pred, label)

            f1_scores.append(f1)
            ious.append(iou)

    val_f1 = np.mean(f1_scores)
    val_iou = np.mean(ious)

    print("Val F1:", val_f1)
    print("Val IoU:", val_iou)

    # Log metrics
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_f1": val_f1,
        "val_iou": val_iou
    })

    # Save best model
    if val_iou > best_iou:

        best_iou = val_iou
        epochs_without_improvement = 0

        torch.save(model.state_dict(), "best_model.pth")

        artifact = wandb.Artifact(
            name="siamese_unet_model",
            type="model"
        )

        artifact.add_file("best_model.pth")
        wandb.log_artifact(artifact)

        print("Saved new best model")

    else:
        epochs_without_improvement += 1
        print(f"No improvement for {epochs_without_improvement} epochs")

    # Early stopping
    if epochs_without_improvement >= patience:
        print("Early stopping triggered")
        break

wandb.finish()

The model also shows stable convergence with the training loss reducing over the epochs. The performance on the validation set increases over the epochs, with the best F1-score achieved at approximately 0.49 and IoU at approximately 0.33. This confirms the successful learning of meaningful representations of the patterns of change in the forest area. However, the performance plateaus at some epochs, which could be due to the difficulty in detecting the extremely sparse change regions. This can be improved with the development of more effective loss functions and the usage of tec

#### Color	Meaning
Set $NIR=True$ to see false color
- Bright red: Healthy vegetation
- Dark brown: Bare soil / cleared land
- Gray: Urban
- Black: Water

In [ ]:
def stretch(img):

    p2, p98 = np.percentile(img, (2, 98))
    img = np.clip((img - p2) / (p98 - p2 + 1e-6), 0, 1)

    return img


def visualize_sample(t1, t2, pred, label, NIR=True):

    t1 = t1.cpu().numpy()
    t2 = t2.cpu().numpy()
    pred = pred.cpu().numpy()
    label = label.cpu().numpy()

    if NIR:
        # False color: NIR-Red-Green
        fc1 = np.stack([t1[3], t1[2], t1[1]], axis=-1)
        fc2 = np.stack([t2[3], t2[2], t2[1]], axis=-1)
    else:
        fc1 = np.stack([t1[2], t1[1], t1[0]], axis=-1)
        fc2 = np.stack([t2[2], t2[1], t2[0]], axis=-1)

    fc1 = stretch(fc1)
    fc2 = stretch(fc2)

    fig, ax = plt.subplots(1,4, figsize=(14,4))

    ax[0].imshow(fc1)
    ax[0].set_title("T1 False Color (NIR-R-G)" if NIR else "T1 (RGB)")
    ax[0].axis("off")

    ax[1].imshow(fc2)
    ax[1].set_title("T2 False Color (NIR-R-G)" if NIR else "T2 (RGB)")
    ax[1].axis("off")

    ax[2].imshow(label, cmap="gray")
    ax[2].set_title("Ground Truth")
    ax[2].axis("off")

    ax[3].imshow(pred, cmap="gray")
    ax[3].set_title("Prediction")
    ax[3].axis("off")

    plt.show()

In [ ]:
# -----------------------
# Load test data
# -----------------------

test_df = pd.read_csv("/kaggle/working/test_split.csv")
root_dir = "/kaggle/input/datasets/suranjandas1990/forest-loss-dataset/dataset"

test_dataset = ChangeDataset(test_df, root_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=True
)


# -----------------------
# Load model
# -----------------------
model = SiameseUNet_ASPP()

state_dict = torch.load("/kaggle/working/best_model.pth", map_location=device)

if "module." in list(state_dict.keys())[0]:
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model = model.to(device)

model.eval();


In [ ]:
# -----------------------
# Run inference
# -----------------------
num_samples = 8
with torch.no_grad():

    for i, (t1, t2, label) in enumerate(test_loader):

        t1 = t1.to(device)
        t2 = t2.to(device)

        pred = model(t1, t2)

        pred = torch.sigmoid(pred)
        pred = (pred > 0.3).float()

        visualize_sample(
            t1[0],
            t2[0],
            pred[0,0],
            label[0].squeeze(),
            NIR=True
        )

        if i >= num_samples + 1:
            break

## Model Improvement techniques

To improve model performance, several enhancements were incorporated into the training pipeline. Data augmentation techniques such as random flipping and rotation were applied to improve generalization. The loss function was refined by combining Dice Loss with Focal Loss to better handle class imbalance and emphasize hard-to-detect regions. Additionally, dynamic threshold optimization was introduced during validation to maximize F1-score, replacing the fixed threshold approach. A learning rate scheduler was also employed to adaptively reduce the learning rate upon performance stagnation, enabling more stable convergence.

In [ ]:
import torchvision.transforms.functional as TF
import random

class ChangeDataset(Dataset):

    def __init__(self, dataframe, root_dir, mean=data_mean, std=data_std, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.root = root_dir
        self.mean = mean
        self.std = std
        self.augment = augment

    def normalize(self, x):
        return (x - self.mean) / (self.std + 1e-6)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]
        region = row["region"]
        fname = row["file"]

        t1 = np.load(f"{self.root}/{region}/t1/{fname}", mmap_mode="r")
        t2 = np.load(f"{self.root}/{region}/t2/{fname}", mmap_mode="r")
        label = np.load(f"{self.root}/{region}/label/{fname}")

        t1 = torch.tensor(t1, dtype=torch.float32)
        t2 = torch.tensor(t2, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32).unsqueeze(0)

        # 🔥 AUGMENTATION
        if self.augment:
            if random.random() > 0.5:
                t1 = TF.hflip(t1)
                t2 = TF.hflip(t2)
                label = TF.hflip(label)

            if random.random() > 0.5:
                t1 = TF.vflip(t1)
                t2 = TF.vflip(t2)
                label = TF.vflip(label)

            if random.random() > 0.5:
                angle = random.choice([90, 180, 270])
                t1 = TF.rotate(t1, angle)
                t2 = TF.rotate(t2, angle)
                label = TF.rotate(label, angle)

        t1 = self.normalize(t1)
        t2 = self.normalize(t2)

        return t1, t2, label

In [ ]:
train_dataset = ChangeDataset(train_df, root_dir, augment=True)
val_dataset   = ChangeDataset(val_df, root_dir, augment=False)

### Focal loss

In [ ]:
class FocalLoss(nn.Module):

    def __init__(self, alpha=0.8, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):

        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)

        focal = self.alpha * (1 - pt) ** self.gamma * bce

        return focal.mean()

In [ ]:
focal = FocalLoss()
dice = DiceLoss()

def loss_fn(pred, target):
    return 0.7 * dice(pred, target) + 0.3 * focal(pred, target)

In [ ]:
def compute_best_threshold(preds, targets):

    thresholds = np.arange(0.1, 0.6, 0.05)
    best_f1 = 0
    best_t = 0.3

    for t in thresholds:
        pred_bin = (preds > t).float()

        tp = (pred_bin * targets).sum()
        fp = (pred_bin * (1 - targets)).sum()
        fn = ((1 - pred_bin) * targets).sum()

        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)

        f1 = 2 * precision * recall / (precision + recall + 1e-6)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

## Updated training loop

In [ ]:
device = "cuda"

model = SiameseUNet_ASPP()

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)

model = model.cuda()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
wandb.init(
    project="Forest loss detection",
    name="unet_aspp_model_improvement",
    config={
        "model": "Siamese_UNet_ASPP",
        "loss": "Dice+Focal",
        "lr": 1e-4,
        "batch_size": 16,
        "epochs": 50
    }
)

In [ ]:
wandb.watch(model, log="all", log_freq=100)

In [ ]:
# ------------------- SETUP -------------------
best_iou = 0
patience = 5
epochs_without_improvement = 0

scaler = GradScaler()

# Optional but recommended
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2,
)

# ------------------- TRAINING -------------------
for epoch in range(50):

    print(f"\nEpoch {epoch+1}/50")

    # -------- TRAIN --------
    model.train()
    train_loss = 0

    for t1, t2, label in tqdm(train_loader):

        t1 = t1.to(device)
        t2 = t2.to(device)
        label = label.to(device)

        optimizer.zero_grad()

        with autocast(device_type="cuda"):
            pred = model(t1, t2)
            loss = loss_fn(pred, label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    print("Train loss:", train_loss)

    # -------- VALIDATION --------
    model.eval()

    f1_scores = []
    ious = []

    with torch.no_grad():

        for t1, t2, label in val_loader:

            t1 = t1.to(device)
            t2 = t2.to(device)
            label = label.to(device)

            with autocast(device_type="cuda"):
                pred = model(t1, t2)

            f1, iou = compute_metrics(pred, label)

            f1_scores.append(f1)
            ious.append(iou)

    val_f1 = np.mean(f1_scores)
    val_iou = np.mean(ious)

    print("Val F1:", val_f1)
    print("Val IoU:", val_iou)

    # -------- LR Scheduler --------
    scheduler.step(val_iou)

    # -------- W&B LOGGING --------
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_f1": val_f1,
        "val_iou": val_iou,
        "lr": optimizer.param_groups[0]['lr']
    })

    # -------- SAVE BEST MODEL --------
    if val_iou > best_iou:

        best_iou = val_iou
        epochs_without_improvement = 0

        torch.save(model.state_dict(), "best_model.pth")

        artifact = wandb.Artifact(
            name="siamese_unet_model",
            type="model"
        )

        artifact.add_file("best_model.pth")
        wandb.log_artifact(artifact)

        print("✅ Saved new best model")

    else:
        epochs_without_improvement += 1
        print(f"⚠️ No improvement for {epochs_without_improvement} epochs")

    # -------- EARLY STOPPING --------
    if epochs_without_improvement >= patience:
        print("🛑 Early stopping triggered")
        break

# ------------------- FINISH -------------------
wandb.finish()